# DeepSeek V3 模型架构实现（MLA + MoE + FP8）

本 Notebook 从零手写 **DeepSeek V3** 的完整推理架构，涵盖：

| 模块 | 说明 |
|---|---|
| **MLA**（Multi-head Latent Attention） | 多头潜在注意力，将 KV 压缩为低维潜在向量，大幅降低 KV Cache |
| **MoE**（Mixture of Experts） | 混合专家，稀疏激活（top-k）提升参数量而不增加计算量 |
| **FP8 量化**（Block-wise FP8 GEMM） | 分块 FP8 矩阵乘法，降低显存并加速推理 |
| **YaRN**（Yet Another RoPE eNhancement） | 旋转位置编码扩展方案，支持 128K 超长序列推理 |
| **分布式并行** | 列/行并行线性层 + 词表并行嵌入，支持多 GPU 张量并行 |

> **运行要求**：L4 GPU（Colab / AutoDL），PyTorch ≥ 2.6，Triton ≥ 3.1，需上传 `kernel.py`

## 一、运行环境准备

本节检查运行本 Notebook 所需的依赖版本，并确认必要文件已就位。

### 1.1 运行前注意事项

In [ ]:
# 运行前置条件说明：
# 1. 需要使用 L4 GPU（推荐 Colab Pro+ 或 AutoDL），官方代码专家数从 64 改为 16（节省显存）
# 2. 注意：本地无 GPU 无法运行；PyTorch 版本不能太低（需支持 RMSNorm）；需安装 Triton ≥ 3.1

# ------------------------------------------------------------------
# 依赖文件 kernel.py 获取方式（从 GitHub 下载）：
#
# kernel.py 来自 DeepSeek-V3 官方仓库，包含 FP8 GEMM 的自定义 Triton Kernel。
# 可通过以下两种方式之一获取：
#
# 方式一：直接用 wget 下载（在 Colab / AutoDL 终端或下方 Shell Cell 中运行）
#   !wget https://raw.githubusercontent.com/deepseek-ai/DeepSeek-V3/main/inference/kernel.py
#
# 方式二：克隆完整仓库后拷贝（网络较慢时推荐先 clone 再取文件）
#   !git clone --depth 1 https://github.com/deepseek-ai/DeepSeek-V3.git
#   !cp DeepSeek-V3/inference/kernel.py .
#
# 下载完成后，确认当前目录存在 kernel.py，再继续执行后续 Cell。
# ------------------------------------------------------------------

### 1.2 检查 Triton 版本与工作目录

确认 Triton ≥ 3.1（FP8 Kernel 要求）以及 `kernel.py` 已上传到当前工作目录。

In [1]:
!pip list|grep triton  # 列出已安装的所有包，通过管道过滤出含 "triton" 的行，确认版本 >= 3.1.0
# 要求结果是 triton 3.1.0 或更高版本；低于 3.1.0 时 FP8 Kernel 无法编译

triton                             3.2.0


In [ ]:
!ls  # 列出当前工作目录下的所有文件，确认 kernel.py 已上传（FP8 GEMM 所需的自定义 Triton Kernel 文件）

kernel.py  __pycache__	sample_data


## 二、全局配置与模型超参数

定义分布式训练全局变量、FP8/BF16 精度选项，以及 DeepSeek V3 所有超参数的配置数据类 `ModelArgs`。

### 2.1 导入依赖库、全局变量与 ModelArgs 超参数配置类

| 全局变量 | 含义 |
|---|---|
| `world_size` | 分布式训练的进程总数，单机推理时为 1 |
| `rank` | 当前进程编号（0 到 world_size-1） |
| `block_size` | FP8 量化的分块大小（128） |
| `gemm_impl` | 矩阵乘法实现方式：`bf16`（默认，直接 BF16 计算）/ `fp8`（量化加速） |
| `attn_impl` | 注意力实现：`naive`（标准展开，易理解）/ `absorb`（KV 吸收，省显存） |

In [ ]:
import math                                          # 数学运算模块，提供 log、floor、ceil、pi 等纯数学函数
from dataclasses import dataclass                    # 数据类装饰器，简化含有大量字段的配置类定义
from typing import Tuple, Optional, Literal          # 类型注解：元组、可选值（X | None）、字面量限定类型

import torch                                         # PyTorch 深度学习框架主模块
from torch import nn                                 # 神经网络模块，提供 Module、Parameter、ModuleList 等基类
import torch.nn.functional as F                     # 无状态函数式接口（linear、embedding、rms_norm 等）
import torch.distributed as dist                    # 分布式训练通信原语（all_reduce、all_gather 等）


print(torch.__version__)                             # 打印当前 PyTorch 版本，确认环境满足 >= 2.6

from kernel import act_quant, weight_dequant, fp8_gemm  # 导入自定义 CUDA Triton Kernel：激活量化、权重反量化、FP8 矩阵乘法

# 定义全局变量
world_size = 1  # 世界大小（用于分布式计算）
rank = 0  # 当前进程的排名
block_size = 128  # 块大小（可能用于矩阵运算优化）
gemm_impl: Literal["bf16", "fp8"] = "bf16"  # 矩阵乘法实现方式（bf16 或 fp8）
attn_impl: Literal["naive", "absorb"] = "absorb"  # 注意力机制实现方式

@dataclass
class ModelArgs:
    """
    用于定义模型参数和超参数的数据类。

    属性：
        max_batch_size (int): 最大批量大小。
        max_seq_len (int): 最大序列长度。
        dtype (Literal["bf16", "fp8"]): 计算数据类型（bf16 或 fp8）。
        vocab_size (int): 词汇表大小。
        dim (int): 模型隐藏层维度。
        inter_dim (int): MLP 层的中间层维度。
        moe_inter_dim (int): MoE 层的中间层维度。
        n_layers (int): Transformer 层的数量。
        n_dense_layers (int): 模型中的全连接层数量。
        n_heads (int): 注意力头的数量。
        n_routed_experts (int): MoE 层中可路由的专家数量。
        n_shared_experts (int): MoE 层中共享的专家数量。
        n_activated_experts (int): MoE 层中每次激活的专家数量。
        n_expert_groups (int): MoE 层中的专家组数量。
        n_limited_groups (int): MoE 路由中的受限组数量。
        score_func (Literal["softmax", "sigmoid"]): MoE 路由的评分函数。
        route_scale (float): MoE 路由评分的缩放因子。
        q_lora_rank (int): 查询（Query）投影的 LoRA 秩。
        kv_lora_rank (int): 键值（Key-Value）投影的 LoRA 秩。
        qk_nope_head_dim (int): 无位置编码的 Query-Key 投影维度。
        qk_rope_head_dim (int): 使用旋转位置编码（RoPE）的 Query-Key 投影维度。
        v_head_dim (int): 值（Value）投影维度。
        original_seq_len (int): 原始序列长度。
        rope_theta (float): 旋转位置编码的基数。
        rope_factor (float): 扩展序列长度的缩放因子。
        beta_fast (int): 快速 Beta 修正因子。
        beta_slow (int): 慢速 Beta 修正因子。
        mscale (float): 扩展注意力的缩放因子。
    """
    max_batch_size: int = 8  # 训练时的最大批次大小
    max_seq_len: int = 4096 * 4  # 允许的最大序列长度（可能用于扩展序列训练），论文中是128K
    dtype: Literal["bf16", "fp8"] = "bf16"  # 计算数据类型，默认使用 bfloat16
    vocab_size: int = 102400  # 词汇表大小
    dim: int = 2048  # 模型隐藏层维度
    inter_dim: int = 10944  # MLP 层的中间层维度
    moe_inter_dim: int = 1408  # MoE 层的中间层维度
    n_layers: int = 27  # Transformer 层数，如果要在本地推理起来，降低这个,论文中61层
    n_dense_layers: int = 1  # 全连接层数（可能用于 MoE 或 MLP 结构）
    n_heads: int = 16  # 注意力头数

    # MoE（Mixture of Experts）相关参数
    n_routed_experts: int = 16  # 可被路由的专家数量,论文里DeepSeekV3用了256个专家,默认是64
    n_shared_experts: int = 2  # 共享专家数量，始终都激活的专家数量，来保证模型的基线性能。论文里只用了1个共享专家
    n_activated_experts: int = 6  # 每次激活的专家数量
    n_expert_groups: int = 1  # 专家组数量（可能用于分组专家路由）
    n_limited_groups: int = 1  # 受限专家组数量
    score_func: Literal["softmax", "sigmoid"] = "softmax"  # MoE 路由评分函数，默认使用 softmax
    route_scale: float = 1.0  # 路由评分的缩放因子

    # MLA（Multi-Level Attention）相关参数
    q_lora_rank: int = 0  # Query 投影的 LoRA 秩（低秩适配）
    kv_lora_rank: int = 512  # Key-Value 投影的 LoRA 秩
    qk_nope_head_dim: int = 128  # 无 RoPE 的 Query-Key 维度
    qk_rope_head_dim: int = 64  # 使用 RoPE 的 Query-Key 维度
    v_head_dim: int = 128  # Value 维度

    # YARN（Yet Another RoPE Network）相关参数
    original_seq_len: int = 4096  # 原始序列长度
    rope_theta: float = 10000.0  # RoPE 旋转位置编码的基数
    rope_factor: float = 40  # 扩展序列长度的缩放因子
    beta_fast: int = 32  # 快速 Beta 修正因子
    beta_slow: int = 1  # 慢速 Beta 修正因子
    mscale: float = 1.0  # 扩展注意力的缩放因子

2.6.0+cu124


## 三、基础模块实现

本节实现 DeepSeek V3 所需的全部底层基础模块：分布式词嵌入层、支持 FP8 量化的线性层及其并行变体、以及 RMSNorm 归一化层。这些模块是后续 MLA 和 MoE 的构建基础。

### 3.1 并行词嵌入层（ParallelEmbedding）与线性变换基类（Linear）

- **`ParallelEmbedding`**：将词表按 `world_size` 均匀分片到各 GPU，每张 GPU 仅存储本分片的嵌入权重；通过 `all_reduce` 将各 GPU 的局部嵌入结果相加，得到完整的词向量
- **`linear`**：底层线性变换函数，自动区分三种情况：标准 BF16 权重直接 `F.linear`、FP8 量化权重先反量化再计算、完整 FP8 路径使用自定义 `fp8_gemm` Kernel
- **`Linear`**：自定义线性层基类，支持 FP8 量化权重（自动附加 `scale` 缩放参数），继承此类实现分布式并行变体

In [4]:
import torch                          # PyTorch 深度学习框架主模块
import torch.nn as nn                 # 神经网络模块，提供 Module、Parameter 等基类
import torch.nn.functional as F      # 无状态函数式接口（embedding 等）
import torch.distributed as dist     # 分布式训练通信原语（all_reduce 等）
from typing import Optional          # 类型注解：Optional[X] 表示该值可为 X 或 None

# 动态获取分布式环境信息；单机推理时 is_initialized() 为 False，两者分别默认为 1 和 0
world_size = dist.get_world_size() if dist.is_initialized() else 1  # 总进程数（GPU 数），单机为 1
rank = dist.get_rank() if dist.is_initialized() else 0              # 当前进程编号，0 表示主进程

class ParallelEmbedding(nn.Module):
    """
    并行嵌入层（ParallelEmbedding），支持分布式训练环境下的词向量分片。

    参数:
        vocab_size (int): 词表大小，即整个模型的词汇总数。
        dim (int): 词向量的维度。

    说明:
        - 词表在多个进程（GPU）之间进行分片，每个进程仅存储词表的一部分。
        - vocab_size 必须能够被 world_size 整除，以确保各 GPU 拥有相同大小的词向量片段。
    """
    def __init__(self, vocab_size: int, dim: int):
        super().__init__()  # 调用 nn.Module 基类构造器，初始化参数管理、hook 等内部状态
        self.vocab_size = vocab_size  # 词表大小
        self.dim = dim  # 词向量维度
        assert vocab_size % world_size == 0, f"词表大小必须能被 world_size 整除 (world_size={world_size})"  # 否则无法均等分片，抛出 AssertionError 提示错误

        self.part_vocab_size = vocab_size // world_size  # 当前 GPU 负责的词表片段大小，类型: int；例如 102400 // 1 = 102400
        self.vocab_start_idx = rank * self.part_vocab_size  # 当前 GPU 词表的起始索引，类型: int；例如 rank=0 时为 0
        self.vocab_end_idx = self.vocab_start_idx + self.part_vocab_size  # 当前 GPU 词表的结束索引（不含），类型: int
        self.weight = nn.Parameter(torch.empty(self.part_vocab_size, self.dim))  # 当前 GPU 的嵌入权重矩阵，形状: (part_vocab_size, dim)，类型: float32

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        并行嵌入层的前向传播。

        参数:
            x (torch.Tensor): 输入张量，包含 token 的索引。

        返回:
            torch.Tensor: 词向量表示。

        处理流程:
            1. 如果使用多 GPU 训练（world_size > 1），检查 token 是否属于当前 GPU 负责的词表范围:
                - 若 token 超出当前 GPU 负责的范围，设为 0（避免索引超界）。
            2. 计算词嵌入。
            3. 若是多 GPU 训练，则进行 all_reduce 操作，将所有 GPU 的嵌入求和（同步）。
        """
        if world_size > 1:  # 多 GPU 模式：词表被分片到不同 GPU，需掩蔽不属于本 GPU 的 token
            mask = (x < self.vocab_start_idx) | (x >= self.vocab_end_idx)  # 布尔掩码：True 表示该 token 不属于本 GPU 词表范围，形状与 x 相同
            x = x - self.vocab_start_idx  # 将全局 token id 转为本地偏移索引，使值落在 [0, part_vocab_size)
            x[mask] = 0  # 越界索引置 0 避免 embedding 查表报错（后续其嵌入值会被清零）

        y = F.embedding(x, self.weight)  # 查表得到词向量，形状: (bsz, seqlen, dim)，越界位置查到 weight[0]（稍后清零）

        if world_size > 1:  # 多 GPU：清零越界嵌入并聚合各 GPU 的局部结果
            y[mask] = 0  # 将非本 GPU 词表范围的 token 的嵌入向量置为全零向量
            dist.all_reduce(y)  # all_reduce 求和：各 GPU 各自贡献其负责词表的部分，求和后每 GPU 得到完整嵌入

        return y  # 返回最终词向量，形状与输入 x 相同，类型: torch.Tensor，值域为嵌入权重的线性组合


def linear(x: torch.Tensor, weight: torch.Tensor, bias: Optional[torch.Tensor] = None) -> torch.Tensor:
    """
    线性变换函数，实现 y = xA^T + b，支持量化权重的计算。

    参数:
        x (torch.Tensor): 输入张量。
        weight (torch.Tensor): 权重张量，可能是量化后的，需要进行解量化处理。
        bias (Optional[torch.Tensor]): 偏置项（可选），默认为 None。

    返回:
        torch.Tensor: 线性变换后的结果。

    说明:
        - 若 weight 不是量化的，则直接调用 F.linear 计算。
        - 若 weight 是量化的（element_size() == 1），则需要先进行解量化，再进行计算。
        - 当 gemm_impl == "bf16" 时，使用 bf16 计算。
        - 其他情况，对 x 进行量化，然后使用 fp8_gemm 计算。
    """
    if weight.element_size() > 1:  # element_size > 1 说明是 BF16/FP16/FP32 权重（非 FP8 量化）
        return F.linear(x, weight, bias)  # 标准矩阵乘法 y = xW^T + b，直接调用 PyTorch 原生实现
    elif gemm_impl == "bf16":  # FP8 权重但指定用 BF16 路径：先反量化权重，再标准计算
        weight = weight_dequant(weight, weight.scale)  # 将 FP8 量化权重还原为 BF16，形状不变，附加的 scale 属性用于反量化
        return F.linear(x, weight, bias)  # 用反量化后的 BF16 权重做标准矩阵乘法
    else:  # FP8 完整路径：对激活值也量化，再调用自定义 Triton FP8 GEMM kernel
        x, scale = act_quant(x, block_size)  # 对输入激活值进行分块 FP8 量化；x 变为 fp8 张量，scale 为对应缩放因子
        y = fp8_gemm(x, scale, weight, weight.scale)  # 调用自定义 Triton kernel 执行 FP8 矩阵乘法，输出 y 为 BF16
        if bias is not None:  # 仅在有偏置时才加，避免额外计算
            y += bias  # 将偏置加到输出上，形状不变（广播机制）
        return y  # 返回 FP8 GEMM 计算结果，形状: (..., out_features)，类型: BF16


class Linear(nn.Module):
    """
    自定义线性层，支持量化权重，并提供可选的偏置项。

    参数:
        in_features (int): 输入特征维度。
        out_features (int): 输出特征维度。
        bias (bool): 是否包含偏置项，默认为 False。
        dtype (可选): 计算数据类型，默认为 torch.bfloat16。

    说明:
        - 如果 weight 是量化的，则需要额外存储 scale 参数。
        - bias 可选，若不使用，则注册为 None。
    """
    dtype = torch.bfloat16  # 默认数据类型

    def __init__(self, in_features: int, out_features: int, bias: bool = False, dtype=None):
        super().__init__()  # 调用 nn.Module 基类构造器
        self.in_features = in_features  # 输入特征维度
        self.out_features = out_features  # 输出特征维度
        self.weight = nn.Parameter(torch.empty(out_features, in_features, dtype=dtype or Linear.dtype))  # 权重矩阵，形状: (out_features, in_features)；优先使用传入 dtype，否则沿用类属性 Linear.dtype

        if self.weight.element_size() == 1:  # FP8 量化权重占 1 字节；BF16/FP32 占 2/4 字节，此条件为 True 时说明启用了 FP8
            scale_out_features = (out_features + block_size - 1) // block_size  # 输出方向的分块数，向上取整，类型: int
            scale_in_features = (in_features + block_size - 1) // block_size    # 输入方向的分块数，向上取整，类型: int
            self.weight.scale = self.scale = nn.Parameter(torch.empty(scale_out_features, scale_in_features, dtype=torch.float32))  # FP8 量化缩放因子，形状: (scale_out, scale_in)，每个分块一个 float32 值
        else:
            self.register_parameter("scale", None)  # 非 FP8 时无需 scale：注册 None 占位，保持接口统一

        if bias:  # 是否使用偏置项（大多数并行线性层默认 False）
            self.bias = nn.Parameter(torch.empty(out_features))  # 偏置向量，形状: (out_features,)，类型: float32
        else:
            self.register_parameter("bias", None)  # 无偏置：注册 None 占位，保持接口一致性

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        线性层的前向传播。

        参数:
            x (torch.Tensor): 输入张量。

        返回:
            torch.Tensor: 经过线性变换的张量。

        说明:
            - 调用 linear 函数，自动处理量化权重和偏置项的计算。
        """
        return linear(x, self.weight, self.bias)  # 委托给 linear 函数，自动处理 BF16 / FP8 路径，返回形状: (..., out_features)


### 3.2 列 / 行并行线性层（ColumnParallelLinear / RowParallelLinear）与均方根归一化（RMSNorm）

- **`ColumnParallelLinear`**：沿输出维度切分，每张 GPU 仅计算总输出的 `1/world_size` 列，适合后接 all-gather
- **`RowParallelLinear`**：沿输入维度切分，每张 GPU 处理部分输入行，计算后通过 `all_reduce` 汇总，适合跟在 ColumnParallel 之后
- **`RMSNorm`**：仅用均方根归一化（无均值减法），相比 LayerNorm 计算量更小，DeepSeek V3 全程使用此归一化

**张量并行标准搭配：必须先列并行、后行并行**

```
完整输入 x（每张 GPU 都有一份）
        │
        ▼
  ColumnParallelLinear          GPU0: W_col_0·x → y_0  (out/N 列)
  （切输出列，无需通信）           GPU1: W_col_1·x → y_1  (out/N 列)
        │
   激活函数（各自独立计算）
        │
        ▼
  RowParallelLinear             GPU0: W_row_0·y_0 → z_0  (部分贡献)
  （切输入行，需通信）             GPU1: W_row_1·y_1 → z_1  (部分贡献)
        │
     all_reduce（求和）
        │
        ▼
  完整输出 z（每张 GPU 都有一份）
```

两者天然「咬合」：列并行的**部分输出**恰好是行并行的**部分输入**，中间无需任何通信，整层只有最后一次 `all_reduce`。
若反过来（先行后列），则行并行需要完整输入，必须先做一次 `all_gather`，变成每层两次通信，效率减半。

**单层 vs 双层线性的并行能力对比**：

| 结构 | 可用并行方式 | 通信次数 | 说明 |
|------|------------|---------|------|
| 单个线性层（只有一个 W） | 只能列并行 **或** 只能行并行，二选一 | 1 次（`all_gather` 或 `all_reduce`） | 列并行：各 GPU 算出部分输出列，需 `all_gather` 拼接成完整输出；行并行：各 GPU 只处理部分输入特征，需提前将完整输入 x 切分后分发给各 GPU |
| 两个连续线性层（W1 → 激活 → W2，即 MLP） | **先列并行（W1）+ 后行并行（W2）** | 1 次（仅最后 `all_reduce`） | 列并行的部分输出恰好是行并行的部分输入，天然咬合，中间零通信 |

因此列并行 + 行并行的搭配**专为 MLP 两层结构设计**，单个线性层无法同时使用两种并行。

**两种通信原语说明**：

| 通信原语 | 行为 | 使用场景 | 示意 |
|---------|------|---------|------|
| `all_gather`（拼接） | 把每张 GPU 上的**局部张量按顺序拼接**，每张 GPU 最终都得到所有 GPU 张量拼成的完整张量 | 列并行：各 GPU 算出部分输出列，需拼接成完整输出 | GPU0:[1,2,3] + GPU1:[4,5,6] → 每张 GPU 都得到 [1,2,3,4,5,6] |
| `all_reduce`（求和） | 把每张 GPU 上的**局部张量对应位置相加**，每张 GPU 最终都得到同一个求和结果 | 行并行：各 GPU 算出部分贡献，需求和得到完整输出 | GPU0:[5,0,1] + GPU1:[0,13,8] → 每张 GPU 都得到 [5,13,9] |

In [ ]:
class ColumnParallelLinear(Linear):
    """
    列并行线性层（Column Parallel Linear），将输出特征分割到多个分布式进程中。

    参数：
        in_features (int): 输入特征的数量。
        out_features (int): 总输出特征数量。
        bias (bool): 是否包含偏置项，默认为 False。
        dtype (optional): 数据类型，默认为 `torch.bfloat16`。
    """
    def __init__(self, in_features: int, out_features: int, bias: bool = False, dtype = None):
        # 确保总输出特征数量可以被世界大小整除，以实现均匀分割
        assert out_features % world_size == 0, f"输出特征数必须能被 world_size 整除 (world_size={world_size})"  # 否则无法均等切分列，抛出 AssertionError
        self.part_out_features = out_features // world_size  # 当前 GPU 负责的输出列数，类型: int；等于 out_features / world_size
        super().__init__(in_features, self.part_out_features, bias, dtype)  # 以局部列数初始化父类 Linear，weight 形状: (part_out_features, in_features)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        列并行线性层的前向传播。

        参数：
            x (torch.Tensor): 输入张量。

        返回：
            torch.Tensor: 经过线性变换后的张量，进行列并行计算。
        """
        # 列并行线性变换：当前 GPU 仅计算 out_features/world_size 列的输出
        y = linear(x, self.weight, self.bias)  # 输出形状: (..., part_out_features)，即总输出的第 rank 段
        return y  # 返回本 GPU 负责的部分输出，调用方需 all-gather 得到完整结果


class RowParallelLinear(Linear):
    """
    行并行线性层（Row Parallel Linear），将输入特征分割到多个分布式进程中。

    参数：
        in_features (int): 总输入特征数量。
        out_features (int): 输出特征的数量。
        bias (bool): 是否包含偏置项，默认为 False。
        dtype (optional): 数据类型，默认为 `torch.bfloat16`。
    """
    def __init__(self, in_features: int, out_features: int, bias: bool = False, dtype = None):
        # 确保输入特征数量可以被世界大小整除，以实现均匀分割
        assert in_features % world_size == 0, f"输入特征数必须能被 world_size 整除 (world_size={world_size})"  # 否则无法均等切分行，抛出 AssertionError
        self.part_in_features = in_features // world_size  # 当前 GPU 负责的输入行数，类型: int；等于 in_features / world_size
        super().__init__(self.part_in_features, out_features, bias, dtype)  # 以局部行数初始化父类 Linear，weight 形状: (out_features, part_in_features)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        行并行线性层的前向传播。

        参数：
            x (torch.Tensor): 输入张量。

        返回：
            torch.Tensor: 经过线性变换后的张量，进行行并行计算。
        """
        # 行并行线性变换：当前 GPU 仅处理 in_features/world_size 行的输入分片
        y = linear(x, self.weight)  # 局部计算结果，形状: (..., out_features)，但值不完整（仅为部分行的贡献）

        if world_size > 1:  # 多 GPU 场景：每张 GPU 只计算了输入的一个分片，需汇总才能得到完整结果
            dist.all_reduce(y)  # 原地加法规约：所有 GPU 的 y 相加，每个 GPU 得到完整的 y

        if self.bias is not None:  # 偏置在 all_reduce 之后加，避免多 GPU 下偏置被重复叠加 world_size 次
            y += self.bias  # 加偏置，广播到输出的每个位置，形状不变

        return y  # 返回完整行并行线性变换结果，形状: (..., out_features)


class RMSNorm(nn.Module):
    """
    均方根归一化（RMSNorm），用于对输入张量进行归一化。

    该方法不同于标准 LayerNorm，不依赖均值，而是基于均方根（RMS）进行归一化。

    参数：
        dim (int): 输入张量的维度。
        eps (float): 用于数值稳定性的 epsilon 值，默认为 1e-6。
    """
    def __init__(self, dim: int, eps: float = 1e-6):
        super().__init__()  # 调用 nn.Module 基类构造器
        self.dim = dim  # 归一化的最后一维大小，类型: int；用于构造 normalized_shape 参数
        self.eps = eps  # 数值稳定性参数，防止除以零，类型: float；默认 1e-6

        # 可学习的缩放参数 γ（与 LayerNorm 中的 weight 作用相同），初始化为全 1 表示恒等变换
        self.weight = nn.Parameter(torch.ones(dim))  # 形状: (dim,)，类型: float32；训练时自动更新

    def forward(self, x: torch.Tensor):
        """
        均方根归一化的前向传播。

        参数：
            x (torch.Tensor): 输入张量，最后一维大小须等于 self.dim。

        返回：
            torch.Tensor: 归一化后的张量，形状与输入 x 完全相同。
        """
        # F.rms_norm 计算: x / sqrt(mean(x²) + eps) * weight，无均值减法
        return F.rms_norm(x, (self.dim,), self.weight, self.eps)  # normalized_shape=(dim,)，沿最后一维归一化；返回形状与 x 相同


## 四、位置编码与多头潜在注意力（MLA）

MLA 是 DeepSeek V3 最核心的创新：通过对 Key-Value 做低秩压缩，将 KV Cache 大幅缩减，同时结合 YaRN 旋转位置编码实现 128K 超长序列支持。

### 4.1 YaRN 旋转位置编码预计算与应用、多头潜在注意力（MLA）

- **`precompute_freqs_cis`**：基于 YaRN 方案预计算 RoPE 复指数频率张量；当序列长度超过训练时长度时，通过线性斜坡平滑插值低频、外推高频，支持 128K 超长序列
- **`apply_rotary_emb`**：将预计算的复指数频率乘到 Q/K 的位置敏感部分，实现旋转位置编码
- **`MLA`（Multi-head Latent Attention）**：DeepSeek V3 核心注意力机制，将 KV 压缩为低维潜在向量（`kv_lora_rank=512`），KV Cache 占用量仅为标准 MHA 的约 1/5
  - `naive` 模式：展开所有 K、V 后计算标准注意力，逻辑最清晰，但约需 48GB 显存
  - `absorb` 模式：将 `wkv_b` 矩阵吸收进 Q 投影，无需展开 KV，显存高效（推荐）

In [ ]:
import torch                  # PyTorch 深度学习框架主模块
import torch.nn as nn         # 神经网络模块（此处主要用于 MLA 类继承）
import math                   # 数学运算模块，提供 log、floor、ceil、pi 等函数
from typing import Optional   # 类型注解：Optional[X] 表示该值可为 X 或 None

# 预计算旋转位置编码的频率值，YaRN
# 该函数用于计算基于旋转位置编码的复指数值
# 主要目的是为了加速计算，避免在每次前向传播时重新计算这些值
def precompute_freqs_cis(args: ModelArgs) -> torch.Tensor:
    """
    预计算旋转位置编码的频率值。

    参数：
        args (ModelArgs): 包含位置编码参数的模型参数。

    返回：
        torch.Tensor: 预计算的复指数值，用于旋转位置编码。
    """
    dim = args.qk_rope_head_dim  # 旋转位置编码的维度,是64
    seqlen = args.max_seq_len  # 最大序列长度
    beta_fast = args.beta_fast  # 快速调整参数
    beta_slow = args.beta_slow  # 缓慢调整参数
    base = args.rope_theta  # 旋转位置编码的基数
    factor = args.rope_factor  # 旋转位置编码的缩放因子

    # 计算旋转位置编码修正维度（内部辅助函数）
    def find_correction_dim(num_rotations, dim, base, max_seq_len):
        """
        计算旋转位置编码的修正维度。

        参数:
            num_rotations (float): 旋转周期数（对应 beta_fast 或 beta_slow）。
            dim (int): RoPE 编码维度。
            base (float): RoPE 基数（rope_theta）。
            max_seq_len (int): 原始训练最大序列长度。

        返回:
            float: 应在哪个维度索引处对频率进行修正。
        """
        return dim * math.log(max_seq_len / (num_rotations * 2 * math.pi)) / (2 * math.log(base))  # 根据旋转周期数反推对应的维度索引，返回 float

    # 计算旋转位置编码修正的起止维度范围（内部辅助函数）
    def find_correction_range(low_rot, high_rot, dim, base, max_seq_len):
        """
        计算旋转位置编码修正范围。

        参数:
            low_rot (float): 低频边界旋转数（对应 beta_fast，频率高于此的维度做插值）。
            high_rot (float): 高频边界旋转数（对应 beta_slow，频率低于此的维度做外推）。
            dim (int): RoPE 编码维度。
            base (float): RoPE 基数。
            max_seq_len (int): 原始训练最大序列长度。

        返回:
            Tuple[int, int]: 修正区间的起止维度索引 (low, high)。
        """
        low = math.floor(find_correction_dim(low_rot, dim, base, max_seq_len))   # 修正区间左端点，向下取整，类型: int
        high = math.ceil(find_correction_dim(high_rot, dim, base, max_seq_len))  # 修正区间右端点，向上取整，类型: int
        return max(low, 0), min(high, dim-1)  # 裁剪到合法维度范围 [0, dim-1]，返回 Tuple[int, int]

    # 计算线性斜坡因子，用于平滑过渡（低频区域=1 保留原始频率，高频区域=0 进行外推）
    def linear_ramp_factor(min, max, dim):
        """
        计算线性斜坡因子（从 0 到 1 的线性插值掩码）。

        参数:
            min (float): 斜坡起始维度索引（该维度处因子为 0）。
            max (float): 斜坡结束维度索引（该维度处因子为 1）。
            dim (int): 输出向量的长度（= dim//2）。

        返回:
            torch.Tensor: 形状为 (dim,) 的 float32 斜坡向量，值域 [0, 1]。
        """
        if min == max:  # 修正区间退化为单点时，分母 (max-min) 将为 0 导致除以零错误
            max += 0.001  # 微调 max，使分母恒为正数，保证计算合法
        linear_func = (torch.arange(dim, dtype=torch.float32) - min) / (max - min)  # 对每个维度索引做线性映射，未裁剪前值域: (-∞, +∞)
        ramp_func = torch.clamp(linear_func, 0, 1)  # 裁剪到 [0, 1]，低于 min 的维度为 0，高于 max 的维度为 1
        return ramp_func  # 返回形状: (dim,)，类型: torch.float32

    # 计算基础 RoPE 频率：θ_i = 1 / (base^(2i/dim))，i 取偶数索引
    freqs = 1.0 / (base ** (torch.arange(0, dim, 2, dtype=torch.float32) / dim))  # 形状: (dim//2,)，类型: float32，值从大到小（低维高频，高维低频）

    if seqlen > args.original_seq_len:  # 只有推理序列超过预训练长度时才需要 YaRN 频率修正，否则沿用标准 RoPE
        low, high = find_correction_range(beta_fast, beta_slow, dim, base, args.original_seq_len)  # 计算旋转位置编码修正起止范围
        smooth = 1 - linear_ramp_factor(low, high, dim // 2)  # 计算平滑过渡的线性斜坡因子
        freqs = freqs / factor * (1 - smooth) + freqs * smooth  # 对超出原始长度部分的频率按修正比例进行融合

    t = torch.arange(seqlen)  # 生成位置索引序列 [0, 1, ..., seqlen-1]，类型: torch.Tensor，形状: (seqlen,)
    freqs = torch.outer(t, freqs)  # 外积：每个位置与每个频率相乘，得到相位矩阵，形状: (seqlen, dim//2)，类型: float32
    freqs_cis = torch.polar(torch.ones_like(freqs), freqs)  # 将相位角转为单位复数 e^{iθ}，形状: (seqlen, dim//2)，类型: complex64
    return freqs_cis  # 返回复数形式的旋转位置编码张量，形状: (max_seq_len, qk_rope_head_dim//2)

# 应用旋转位置编码到输入张量
# 该函数使用预计算的复指数值对输入进行旋转编码

def apply_rotary_emb(x: torch.Tensor, freqs_cis: torch.Tensor) -> torch.Tensor:
    """
    应用旋转位置编码到输入张量。

    参数：
        x (torch.Tensor): 输入张量。
        freqs_cis (torch.Tensor): 预计算的复指数值。

    返回：
        torch.Tensor: 旋转编码后的张量。
    """
    dtype = x.dtype  # 保存输入的原始数据类型（如 bfloat16），用于最终恢复精度
    # view_as_complex：实数张量 → 复数张量（零拷贝视图）
    #   RoPE 的核心思想：对每对相邻维度 (x0, x1) 做 2D 旋转，等价于复数乘法：
    #   (x0 + i·x1) × (cosθ + i·sinθ) = (x0·cosθ - x1·sinθ) + i·(x0·sinθ + x1·cosθ)
    #   步骤：先将最后一维两两配对 (..., dim) → (..., dim//2, 2)，再视为复数 (..., dim//2)
    #   例：[1.0, 2.0, 3.0, 4.0] → [[1,2],[3,4]] → [1+2j, 3+4j]
    x = torch.view_as_complex(x.float().view(*x.shape[:-1], -1, 2))  # 实数 (..., dim) → 复数 (..., dim//2)，类型从 float32 变为 complex64
    freqs_cis = freqs_cis.view(1, x.size(1), 1, x.size(-1))  # 广播调整形状至 (1, seqlen, 1, dim//2)，便于与 x 逐元素复数相乘
    # view_as_real：复数张量 → 实数张量（零拷贝视图），与 view_as_complex 互为逆操作
    #   x * freqs_cis：复数乘法，等价于对每对 (x0,x1) 施加 2D 旋转矩阵，比手写矩阵乘更高效
    #   view_as_real：复数 (..., dim//2) → 实数 (..., dim//2, 2)，将每个复数展开为 [实部, 虚部]
    #   例：[5+6j, 7+8j] → [[5,6],[7,8]] → flatten → [5,6,7,8]
    #   flatten(3)：将最后两维合并，shape 恢复为原始 (..., dim)
    y = torch.view_as_real(x * freqs_cis).flatten(3)  # 复数乘法旋转后展平回实数，形状恢复为 (..., dim)，类型: float32
    return y.to(dtype)  # 将结果转换回输入的原始数据类型，形状与输入 x 相同

# 多头注意力层（MLA）  多头隐性注意力  多头潜在注意力
# 该类实现了标准的多头注意力机制，并结合了旋转位置编码
class MLA(nn.Module):
    """
    多头注意力层（MLA）。

    属性:
        dim (int): 输入特征的维度。
        n_heads (int): 注意力头的数量。
        n_local_heads (int): 分布式系统中用于局部注意力的头数量。
        q_lora_rank (int): 查询低秩投影的秩。
        kv_lora_rank (int): 键值低秩投影的秩。
        qk_nope_head_dim (int): 非位置查询/键投影的维度。
        qk_rope_head_dim (int): 旋转位置查询/键投影的维度。
        qk_head_dim (int): 查询/键投影的总维度。
        v_head_dim (int): 值投影的维度。
        softmax_scale (float): 注意力计算中Softmax的缩放因子。
    """
    def __init__(self, args: ModelArgs):
        super().__init__()  # 调用 nn.Module 基类构造器，初始化参数管理机制
        # 初始化各个参数
        self.dim = args.dim  # 输入的特征维度
        self.n_heads = args.n_heads  # 注意力头的数量
        self.n_local_heads = args.n_heads // world_size  # 分布式环境中的局部注意力头数
        self.q_lora_rank = args.q_lora_rank  # 查询低秩投影的秩
        self.kv_lora_rank = args.kv_lora_rank  # 键值低秩投影的秩
        self.qk_nope_head_dim = args.qk_nope_head_dim  # 非位置查询/键的维度
        self.qk_rope_head_dim = args.qk_rope_head_dim  # 旋转位置查询/键的维度
        self.qk_head_dim = args.qk_nope_head_dim + args.qk_rope_head_dim  # 查询/键投影的总维度
        self.v_head_dim = args.v_head_dim  # 值投影的维度

        if self.q_lora_rank == 0:  # q_lora_rank=0 表示直接全秩投影（DeepSeekV2 风格），非0 表示低秩分解节省参数（DeepSeekV3 完整版）
            self.wq = ColumnParallelLinear(self.dim, self.n_heads * self.qk_head_dim)  # 直接投影：dim → n_heads * qk_head_dim (2048→16*192=3072)
        else:
            self.wq_a = Linear(self.dim, self.q_lora_rank)  # 低秩投影的第一部分，不为0，为512，先从2048变为512
            self.q_norm = RMSNorm(self.q_lora_rank)  # 低秩投影的标准化
            self.wq_b = ColumnParallelLinear(self.q_lora_rank, self.n_heads * self.qk_head_dim)  # 低秩投影的第二部分

        # 键值投影和标准化
        self.wkv_a = Linear(self.dim, self.kv_lora_rank + self.qk_rope_head_dim)  # 键值低秩投影
        self.kv_norm = RMSNorm(self.kv_lora_rank)  # 键值的标准化
        self.wkv_b = ColumnParallelLinear(self.kv_lora_rank, self.n_heads * (self.qk_nope_head_dim + self.v_head_dim))  # 键值投影
        self.wo = RowParallelLinear(self.n_heads * self.v_head_dim, self.dim)  # 输出投影
        self.softmax_scale = self.qk_head_dim ** -0.5  # Softmax缩放因子

        if args.max_seq_len > args.original_seq_len:  # 只有超长序列推理时才需要 YaRN 修正 softmax_scale，防止注意力得分因维度偏移而过大
            mscale = 0.1 * args.mscale * math.log(args.rope_factor) + 1.0  # YaRN 注意力缩放修正系数，类型: float；rope_factor 越大，修正越强
            self.softmax_scale = self.softmax_scale * mscale * mscale       # 对 softmax_scale 乘以 mscale²，补偿超长序列下注意力得分的分布偏移

        if attn_impl == "naive":  # naive 模式缓存完整展开的 K/V；absorb 模式仅缓存 KV 潜在向量（显存更省）
            # naive 模式：存储展开后的完整 K 和 V（显存大，但逻辑直观）
            self.register_buffer("k_cache", torch.zeros(args.max_batch_size, args.max_seq_len, self.n_local_heads, self.qk_head_dim), persistent=False)  # K 缓存，形状: (max_batch, max_seq, n_heads, 192)
            self.register_buffer("v_cache", torch.zeros(args.max_batch_size, args.max_seq_len, self.n_local_heads, self.v_head_dim), persistent=False)    # V 缓存，形状: (max_batch, max_seq, n_heads, 128)
        else:
            # absorb 模式：仅存储 KV 潜在向量和 K 的位置编码（显存小，推理高效）
            self.register_buffer("kv_cache", torch.zeros(args.max_batch_size, args.max_seq_len, self.kv_lora_rank), persistent=False)         # KV 潜在向量缓存，形状: (max_batch, max_seq, 512)
            self.register_buffer("pe_cache", torch.zeros(args.max_batch_size, args.max_seq_len, self.qk_rope_head_dim), persistent=False)     # K 位置编码缓存，形状: (max_batch, max_seq, 64)

    def forward(self, x: torch.Tensor, start_pos: int, freqs_cis: torch.Tensor, mask: Optional[torch.Tensor]):
        """
        多头注意力层的前向传播。

        参数:
            x (torch.Tensor): 输入张量，形状为(batch_size, seq_len, dim)。
            start_pos (int): 缓存的起始位置。
            freqs_cis (torch.Tensor): 预计算的旋转嵌入的复数指数值。
            mask (Optional[torch.Tensor]): 掩码张量，用于排除某些位置的注意力计算。

        返回:
            torch.Tensor: 输出张量，形状与输入相同。
        """
        bsz, seqlen, _ = x.size()  # 获取输入张量的batch size和序列长度
        end_pos = start_pos + seqlen  # 计算结束位置

        if self.q_lora_rank == 0:  # 判断 Q 使用直接投影还是低秩分解路径
            q = self.wq(x)  # 直接投影：(bsz, seqlen, dim) → (bsz, seqlen, n_heads*qk_head_dim)
        else:
            q = self.wq_b(self.q_norm(self.wq_a(x)))  # 使用低秩投影和标准化计算查询

        # 将 Q 展开为多头格式，便于后续按头拆分 nope 和 rope 两部分
        q = q.view(bsz, seqlen, self.n_local_heads, self.qk_head_dim)  # 重塑形状：(bsz, seqlen, n_local_heads, qk_head_dim=192)
        print(f"q shape: {q.shape}")  # 调试输出：确认形状为 (batch_size, seq_len, 16, 192)
        q_nope, q_pe = torch.split(q, [self.qk_nope_head_dim, self.qk_rope_head_dim], dim=-1)  # 按 [128, 64] 切分最后一维：q_nope 用于语义点积，q_pe 施加 RoPE
        # 调试输出：确认 q_nope 形状 (bsz,seqlen,16,128) 和 q_pe 形状 (bsz,seqlen,16,64)
        print(f"q_nope shape: {q_nope.shape}, q_pe shape: {q_pe.shape}")  # q_nope 用于语义相似度计算，q_pe 施加旋转位置编码
        q_pe = apply_rotary_emb(q_pe, freqs_cis)  # 对 Q 的位置敏感部分施加 RoPE，使模型感知 token 的绝对/相对位置，形状不变: (bsz,seqlen,n_heads,64)

        # 计算键值（kv）：先将输入投影到低维潜在空间（kv_lora_rank）+ 位置编码（qk_rope_head_dim）
        kv = self.wkv_a(x)  # 线性投影，输出形状: (bsz, seqlen, kv_lora_rank + qk_rope_head_dim) = (bsz, seqlen, 576)
        print("kv shape:", kv.shape)  # 调试输出：确认形状为 (batch_size, seq_len, 576)
        kv, k_pe = torch.split(kv, [self.kv_lora_rank, self.qk_rope_head_dim], dim=-1)  # 拆分为 KV 潜在向量 (bsz,seqlen,512) 和 K 位置编码 (bsz,seqlen,64)
        print(f"kv shape: {kv.shape}, k_pe shape: {k_pe.shape}")  # 调试输出：确认 kv (bsz,seqlen,512)、k_pe (bsz,seqlen,64)
        k_pe = apply_rotary_emb(k_pe.unsqueeze(2), freqs_cis)  # unsqueeze(2) 增加 head 维度→(bsz,seqlen,1,64)，施加 RoPE 后形状不变

        if attn_impl == "naive":  # naive=标准完整展开（逻辑清晰，需较大显存），absorb=矩阵吸收优化（节省显存）
            q = torch.cat([q_nope, q_pe], dim=-1)  # 拼接查询的非位置部分和位置编码部分，形状: (bsz, seqlen, 16, 192)
            kv = self.wkv_b(self.kv_norm(kv))  # 对键值进行标准化 kv shape(bs,seqlen,16*256)
            kv = kv.view(bsz, seqlen, self.n_local_heads, self.qk_nope_head_dim + self.v_head_dim)  # 重塑为多头格式，形状: (bsz, seqlen, n_heads, qk_nope+v_head_dim=256)
            k_nope, v = torch.split(kv, [self.qk_nope_head_dim, self.v_head_dim], dim=-1)  # 按 qk_nope_head_dim(128) 和 v_head_dim(128) 切分：k_nope (bsz,seqlen,16,128)，v (bsz,seqlen,16,128)
            k = torch.cat([k_nope, k_pe.expand(-1, -1, self.n_local_heads, -1)], dim=-1)  # 拼接键的非位置部分和位置编码部分
            print(f"k shape: {k.shape}")  # 调试输出：确认 k 形状为 (bsz, seqlen, 16, 192=128+64)
            self.k_cache[:bsz, start_pos:end_pos] = k  # 将当前步的 K 写入 KV Cache，供后续 token 复用（增量推理加速）
            self.v_cache[:bsz, start_pos:end_pos] = v  # 将当前步的 V 写入 KV Cache，形状: (bsz, end_pos, n_heads, v_head_dim)
            # Q·K^T 注意力得分：对所有历史位置（0~end_pos）计算点积相似度
            # torch.einsum(pattern, A, B)：爱因斯坦求和约定，核心规则：
            #   出现在输入下标中但【不出现在输出下标】中的维度 → 自动对该维度逐元素相乘后求和（收缩消掉）
            #   出现在输出下标中的维度                         → 保留该维度，原样传递
            # 下标含义：b=batch, s=query序列位置, h=注意力头, d=每头维度, t=key序列位置（历史缓存长度）
            # "bshd,bthd->bsht" 各维度状态：
            #   b: 输入1有、输入2有、输出有 → 保留（batch 维，原样传递）
            #   s: 输入1有、输入2无、输出有 → 保留（query 序列位置）
            #   t: 输入1无、输入2有、输出有 → 保留（key 历史位置）
            #   h: 输入1有、输入2有、输出有 → 保留（注意力头）
            #   d: 输入1有、输入2有、输出无 → 【求和消掉】对 head_dim 做点积
            # 输入 q          形状: (bsz, seqlen, n_heads, head_dim)   ← 当前查询
            # 输入 k_cache    形状: (bsz, end_pos, n_heads, head_dim)  ← 历史键缓存
            # 输出 scores     形状: (bsz, seqlen, n_heads, end_pos)    ← 每个查询对每个历史位置的注意力得分
            # 计算：scores[b,s,h,t] = Σ_d( q[b,s,h,d] × k[b,t,h,d] ) ← 向量点积，衡量 Q 与 K 的相似程度
            # 求和的本质：把两个 head_dim 维向量对应位置相乘再加总，即向量点积
            # ⚠️ K 被隐式转置：k_cache 下标为 t,d（t 在前、d 在后），求和维度 d 在 K 的第二维
            #   → 等价于 Q @ Kᵀ，即 (seqlen×head_dim) @ (head_dim×end_pos) = (seqlen×end_pos)
            #   判断规则：共享下标（d）在某输入的末尾且不出现在输出 → 该输入被转置
            #   对比：若下标为 "bshd,bhtd->bsht"（d 在 K 的 第一维）则等价于 Q @ K（无转置）
            # 等价显式写法：q @ k_cache.transpose(-1, -2)
            scores = torch.einsum("bshd,bthd->bsht", q, self.k_cache[:bsz, :end_pos]) * self.softmax_scale  # Q·K 点积注意力得分，乘缩放因子 1/√head_dim 防止数值过大，形状: (bsz, seqlen, n_heads, end_pos)
        else:
            # ■ absorb（吸收）模式核心思路：
            #   标准 MLA 注意力得分推导：
            #     Q 和 K 均由 nope（语义）和 pe（位置）两段拼接而成：
            #       Q = [Q_nope | Q_pe]   形状: (bsz, seq, heads, 128+64=192)
            #       K = [K_nope | K_pe]   形状: (bsz, seq, heads, 128+64=192)
            #     两向量点积可按分段拆开（拼接向量的点积 = 对应段点积之和）：
            #       [a | b] · [c | d] = a·c + b·d
            #     因此：
            #       scores = Q @ K^T = [Q_nope | Q_pe] @ [K_nope | K_pe]^T
            #              = Q_nope @ K_nope^T + Q_pe @ K_pe^T
            #                ↑ 语义相似度（无位置信息）  ↑ 位置相似度（RoPE 旋转后的点积）
            #   其中 K_nope = kv_norm(kv_latent) @ wkv_b_K^T  （wkv_b_K 是 wkv_b 权重的 K 行分区）
            #
            #   kv_latent 是什么：
            #     MLA 不直接计算完整 K/V，而是先把输入 x 通过下投影矩阵 wkv_a 压缩成一个低维「胶囊」：
            #       x (dim=2048) → wkv_a → [kv_latent (512维) | k_pe (64维)]  共 576 维
            #     kv_latent 这 512 维向量同时编码了 K 和 V 的全部语义信息，K 和 V 都从它通过上投影 wkv_b 展开：
            #       kv_latent (512) → wkv_b → 完整 K (16头×128) + 完整 V (16头×128)
            #     它是 MLA 节省 KV Cache 显存的核心：
            #       标准 MHA 每 token 缓存 K+V = 16头×(128+128) = 4096 维
            #       MLA absorb 每 token 只缓存 kv_latent(512) + k_pe(64) = 576 维，节省约 7 倍
            #
            #   代入后：Q_nope @ K_nope^T
            #         = Q_nope @ (kv_latent @ wkv_b_K^T)^T
            #         = Q_nope @ wkv_b_K @ kv_latent^T      ← 利用矩阵乘法结合律重排
            #         = (Q_nope @ wkv_b_K) @ kv_latent^T    ← 把 wkv_b_K「吸收」进 Q
            #   好处：KV Cache 只需缓存 kv_latent(512) + pe_cache(64) = 576-dim
            #         而非展开后的完整 K+V：k_cache(16头×192) + v_cache(16头×128) = 3072+2048 = 5120-dim
            #         显存节省比 ≈ 5120 / 576 ≈ 8.9 倍

            # ① 取出 wkv_b 权重矩阵（处理 FP8 量化与普通精度两种情况）
            # self.wkv_b.scale 为 None 表示普通精度（BF16/FP32），直接取 .weight
            # 否则为 FP8 量化权重，需先调用 weight_dequant 反量化恢复为高精度浮点
            # wkv_b 形状: (n_local_heads*(qk_nope_head_dim + v_head_dim), kv_lora_rank)
            #           = (16*(128+128), 512) = (4096, 512)
            # 该矩阵同时编码了 K 的语义投影（前 qk_nope_head_dim 行/头）和 V 的投影（后 v_head_dim 行/头）
            wkv_b = self.wkv_b.weight if self.wkv_b.scale is None else weight_dequant(self.wkv_b.weight, self.wkv_b.scale, block_size)  # 返回 torch.Tensor，形状: (n_heads*(qk_nope+v_head_dim), kv_lora_rank) = (4096, 512)

            # ② 将 wkv_b 重塑为「按头分组」的三维张量，方便后续按头做矩阵运算
            # view 中 -1 由 PyTorch 自动推断，= qk_nope_head_dim + v_head_dim = 128+128 = 256
            # 重塑后可按 wkv_b[h] 取第 h 个头的子矩阵，形状: (256, 512)
            # 其中 wkv_b[h, :qk_nope_head_dim, :] 是第 h 头的 K 投影矩阵，形状: (128, 512)
            #      wkv_b[h, -v_head_dim:, :]      是第 h 头的 V 投影矩阵，形状: (128, 512)
            wkv_b = wkv_b.view(self.n_local_heads, -1, self.kv_lora_rank)  # 返回 torch.Tensor，形状: (n_local_heads, qk_nope_head_dim+v_head_dim, kv_lora_rank) = (16, 256, 512)

            # ③ 将 wkv_b 的 K 分区吸收进 q_nope（核心变换）
            # 数学含义：q_nope_absorbed[b,s,h,:] = Σ_d( q_nope[b,s,h,d] × wkv_b[h,d,:] )
            #   即对每个头 h，用 q_nope 的 head_dim 维向量左乘 wkv_b_K 矩阵
            #   结果从 qk_nope_head_dim(128) 维压缩/投影到 kv_lora_rank(512) 维
            # einsum 下标含义：b=batch, s=query位置, h=头, d=qk_nope_head_dim(128, 求和消掉), c=kv_lora_rank(512, 保留)
            # wkv_b[:, :self.qk_nope_head_dim] 取 K 分区，形状: (16, 128, 512)
            # 吸收后 q_nope 与 kv_cache（潜在向量空间）处于同一向量空间，可直接点积
            q_nope = torch.einsum("bshd,hdc->bshc", q_nope, wkv_b[:, :self.qk_nope_head_dim])  # 返回 torch.Tensor，形状: (bsz, seqlen, n_local_heads, kv_lora_rank) = (bsz, seqlen, 16, 512)

            # ④ 将当前 token 的归一化 KV 潜在向量写入 kv_cache（仅缓存压缩态，不展开）
            # self.kv_norm 是 RMSNorm，对 kv 潜在向量做归一化，稳定训练/推理数值
            # kv 形状: (bsz, seqlen, kv_lora_rank=512)；[:bsz] 防止 batch 越界，[start_pos:end_pos] 写入当前位置
            # 与 naive 模式对比：naive 缓存展开后的 K (bsz,seq,16头,128) + V (bsz,seq,16头,128)，共 16×256 = 4096 维
            #                   absorb 仅缓存潜在向量 512 维，节省约 8 倍显存
            self.kv_cache[:bsz, start_pos:end_pos] = self.kv_norm(kv)  # 写入后 kv_cache 形状: (max_batch, end_pos, kv_lora_rank=512)

            # ⑤ 将当前 token 的 K 位置编码写入 pe_cache
            # k_pe 在进入此分支前已施加 RoPE，形状: (bsz, seqlen, 1, qk_rope_head_dim=64)
            # squeeze(2) 去掉多余的 head 维度（K 的 RoPE 部分所有头共享同一套位置编码）
            # → 形状变为 (bsz, seqlen, 64)，存入 pe_cache
            self.pe_cache[:bsz, start_pos:end_pos] = k_pe.squeeze(2)  # 写入后 pe_cache 形状: (max_batch, end_pos, qk_rope_head_dim=64)

            # ⑥ 计算注意力得分（分两部分相加，对应 nope 和 rope 两个子空间）
            # 完整得分 = 语义部分（nope）+ 位置部分（rope）
            #
            # 【语义部分（nope）】
            # einsum "bshc,btc->bsht"：
            #   下标：b=batch, s=query位置, h=头, c=kv_lora_rank(512, 求和消掉), t=历史key位置
            #   含义：q_nope_absorbed[b,s,h,:] 与 kv_cache[b,t,:] 做点积
            #   等价于 Q_nope @ wkv_b_K @ kv_latent^T，即展开后的 Q_nope @ K_nope^T
            #   注意：kv_cache 无头维度（各头共享同一套潜在向量），广播到所有 h 头
            # 输入 q_nope    形状: (bsz, seqlen, 16, 512)
            # 输入 kv_cache  形状: (bsz, end_pos, 512)
            # 输出           形状: (bsz, seqlen, 16, end_pos)
            #
            # 【位置部分（rope）】
            # einsum "bshr,btr->bsht"：
            #   下标：b=batch, s=query位置, h=头, r=qk_rope_head_dim(64, 求和消掉), t=历史key位置
            #   含义：q_pe[b,s,h,:] 与 pe_cache[b,t,:] 做点积，衡量 query 与历史 token 的位置相似度
            #   pe_cache 同样无头维度，广播到所有 h 头
            # 输入 q_pe      形状: (bsz, seqlen, 16, 64)
            # 输入 pe_cache  形状: (bsz, end_pos, 64)
            # 输出           形状: (bsz, seqlen, 16, end_pos)
            #
            # 两部分相加后乘 softmax_scale（= 1/√qk_head_dim）防止点积数值过大导致 softmax 梯度消失
            scores = (torch.einsum("bshc,btc->bsht", q_nope, self.kv_cache[:bsz, :end_pos]) +  # 语义得分，形状: (bsz, seqlen, n_local_heads, end_pos)
                      torch.einsum("bshr,btr->bsht", q_pe, self.pe_cache[:bsz, :end_pos])) * self.softmax_scale  # 位置得分相加后乘缩放因子，最终形状: (bsz, seqlen, n_local_heads, end_pos)

        if mask is not None:  # 因果掩码不为 None 时（seqlen>1 的 prefill 阶段），防止当前 token 看到未来位置
            scores += mask.unsqueeze(1)  # mask.unsqueeze(1) 形状: (seqlen, 1, seqlen)→广播到 (bsz, n_heads, seqlen, seqlen)

        # 对最后一维（所有历史位置）做 Softmax，得到注意力权重
        scores = scores.softmax(dim=-1, dtype=torch.float32).type_as(x)  # 用 float32 计算 softmax 防止溢出，再转回输入精度；形状不变: (bsz, seqlen, n_heads, end_pos)

        if attn_impl == "naive":  # naive 模式：直接用注意力权重对缓存的 V 加权求和
            x = torch.einsum("bsht,bthd->bshd", scores, self.v_cache[:bsz, :end_pos])  # 标准注意力聚合，输出形状: (bsz, seqlen, n_heads, v_head_dim)
        else:
            x = torch.einsum("bsht,btc->bshc", scores, self.kv_cache[:bsz, :end_pos])  # absorb 模式：先与 KV 潜在向量聚合，形状: (bsz, seqlen, n_heads, kv_lora_rank)
            x = torch.einsum("bshc,hdc->bshd", x, wkv_b[:, -self.v_head_dim:])  # 再用 wkv_b 的 V 部分展开，形状: (bsz, seqlen, n_heads, v_head_dim)

        # 将各注意力头的输出拼接后，通过输出投影层映射回原始隐藏维度
        x = self.wo(x.flatten(2))  # flatten(2) 将 (bsz, seqlen, n_heads, v_head_dim) 展平为 (bsz, seqlen, n_heads*v_head_dim)；wo 输出形状: (bsz, seqlen, dim)
        return x  # 返回注意力层输出，形状: (bsz, seqlen, dim)，与输入 x 形状相同



## 五、前馈网络与混合专家（MoE）基础组件

DeepSeek V3 的 MoE 层由三部分协同工作：Gate（路由）、Expert（专家网络）和 SharedExpert（共享专家）。本节先实现 MLP 和各子模块，下节再组合为完整 MoE。

### 5.1 多层感知机（MLP）、MoE 门控机制（Gate）与专家网络（Expert）

- **`MLP`**：标准 SwiGLU 前馈网络，计算公式为 `w2(silu(w1(x)) * w3(x))`，用于模型前几个稠密 Transformer 层
- **`Gate`**：接收每个 token 的隐藏向量，计算其与各专家的亲和度分数，选出 top-k 个专家并返回路由权重
- **`Expert`**：结构与 MLP 相同的独立前馈网络，但每个专家参数独立、中间维度更小（`moe_inter_dim`），通过稀疏激活实现参数量远大于计算量

In [ ]:
import torch                          # PyTorch 深度学习框架主模块
import torch.nn as nn                 # 神经网络模块，提供 Module、Parameter 等基类
import torch.nn.functional as F      # 无状态函数式接口（silu 激活函数等）
from typing import Tuple             # 类型注解：Tuple[A, B] 表示包含 A 和 B 两个元素的元组

class MLP(nn.Module):
    """
    多层感知机（MLP），用于前馈计算。

    该模块包含三个线性变换层，分别是 w1、w2 和 w3，用于特征变换和计算。

    属性:
        w1 (nn.Module): 线性层，用于从输入层到隐藏层的转换。
        w2 (nn.Module): 线性层，用于从隐藏层到输出层的转换。
        w3 (nn.Module): 额外的线性层，用于特征变换。
    """
    def __init__(self, dim: int, inter_dim: int):
        """
        初始化 MLP 层。

        参数:
            dim (int): 输入和输出的维度（维度保持一致）。
            inter_dim (int): 隐藏层的维度。
        """
        super().__init__()  # 调用 nn.Module 基类构造器
        self.w1 = ColumnParallelLinear(dim, inter_dim)  # 门控分支 W₁：dim → inter_dim，输出经 SiLU 激活
        self.w2 = RowParallelLinear(inter_dim, dim)     # 输出投影 W₂：inter_dim → dim，将激活后的特征映射回原始维度
        self.w3 = ColumnParallelLinear(dim, inter_dim)  # 值分支 W₃：dim → inter_dim，与 silu(w1(x)) 相乘实现门控机制

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        MLP 的前向计算。

        参数:
            x (torch.Tensor): 输入张量，形状为 (batch_size, dim)。

        返回:
            torch.Tensor: 经过 MLP 计算后的输出张量，形状为 (batch_size, dim)。
        """
        return self.w2(F.silu(self.w1(x)) * self.w3(x))  # SwiGLU 激活：silu(w1(x)) 作门控，w3(x) 作值；乘积后经 w2 投影，输出形状: (..., dim)


class Gate(nn.Module):
    """
    用于 Mixture-of-Experts（MoE）模型的门控机制（Gating Mechanism）。

    该模块用于在多个专家（Expert）之间进行路由选择，决定每个输入数据应该被送到哪些专家进行计算。

    属性:
        dim (int): 输入特征的维度。
        topk (int): 每个输入激活的专家数（选择 top-k 个专家）。
        n_groups (int): 专家被划分的组数（用于路由分组）。
        topk_groups (int): 每个输入路由到的专家组数。
        score_func (str): 计算分数的函数（可选 "softmax" 或 "sigmoid"）。
        route_scale (float): 路由权重的缩放因子。
        weight (torch.nn.Parameter): 可训练参数，表示门控网络的权重矩阵。
        bias (Optional[torch.nn.Parameter]): 可选的偏置项，仅当输入维度为 7168 时存在。
    """
    def __init__(self, args: ModelArgs):
        """
        初始化 Gate 模块。

        参数:
            args (ModelArgs): 传入的模型参数对象，包含 MoE 相关的超参数。
        """
        super().__init__()  # 调用 nn.Module 基类构造器
        self.dim = args.dim                           # 输入特征的维度，类型: int；例如 2048
        self.topk = args.n_activated_experts          # 每个 token 激活的专家数（top-k），类型: int；例如 6
        self.n_groups = args.n_expert_groups          # 专家分组数量，简化版为 1（完整版为 8），类型: int
        self.topk_groups = args.n_limited_groups      # 分组路由时选择的组数，简化版为 1，类型: int
        self.score_func = args.score_func             # 亲和度得分函数："softmax" 或 "sigmoid"，类型: str
        self.route_scale = args.route_scale           # 路由权重全局缩放系数，类型: float；例如 1.0

        # 可训练的门控权重矩阵，形状: (n_routed_experts, dim)，每行是一个专家的原型向量
        self.weight = nn.Parameter(torch.empty(args.n_routed_experts, args.dim))  # 类型: nn.Parameter，参与梯度更新

        # 仅在完整版（dim=7168）时使用可学习偏置项；偏置仅影响路由决策，不参与权重归一化
        self.bias = nn.Parameter(torch.empty(args.n_routed_experts)) if self.dim == 7168 else None  # 形状: (n_routed_experts,) 或 None

    def forward(self, x: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:
        """
        计算门控权重，并确定选择哪些专家进行计算。

        参数:
            x (torch.Tensor): 输入张量，形状为 (bsz*seqlen, dim)；在传入 Gate 前已由 MoE 层将 (bsz,seqlen,dim) flatten 为二维。

        返回:
            Tuple[torch.Tensor, torch.Tensor]:
                - 选择的专家权重 (bsz*seqlen, topk)  # 注：x 在传入前已 flatten，第一维为 bsz*seqlen 而非 batch_size
                - 选择的专家索引 (bsz*seqlen, topk)  # 每个 token 被路由到 topk 个专家的索引
        """
        print(f"Gate x shape: {x.shape}")  # 调试输出：确认输入形状为 (bsz*seqlen, hidden_dim)，即每行是一个 token 的特征向量
        scores = linear(x, self.weight)  # 计算每个 token 与所有专家的亲和度得分，形状: (bsz*seqlen, n_routed_experts)

        if self.score_func == "softmax":  # softmax 路径：所有专家竞争性分配概率，得分之和为 1
            scores = scores.softmax(dim=-1, dtype=torch.float32)  # Softmax 归一化，使用 float32 防止精度溢出，形状不变
        else:
            scores = scores.sigmoid()  # Sigmoid 独立激活：每个专家得分独立映射到 (0,1)，和不为 1

        original_scores = scores  # 保存未加偏置的原始分数，用于后续计算专家权重（偏置仅影响路由决策，不影响权重值）

        if self.bias is not None:  # 有可学习偏置时加到得分上，调整路由倾向（仅影响 topk 选择，不影响权重计算）
            scores = scores + self.bias  # 加偏置后的 scores 仅用于专家路由选择，形状不变

        if self.n_groups > 1:  # 简化版 n_groups=1 跳过此块；完整版分组路由降低负载不均衡风险
            scores = scores.view(x.size(0), self.n_groups, -1)  # 重塑为 (bsz*seqlen, n_groups, n_experts_per_group)

            if self.bias is None:  # 无可学习偏置时，以组内最高得分代表该组质量
                group_scores = scores.amax(dim=-1)           # 取每组最大得分，形状: (bsz*seqlen, n_groups)
            else:
                group_scores = scores.topk(2, dim=-1)[0].sum(dim=-1)  # 取每组 top-2 得分之和，形状: (bsz*seqlen, n_groups)

            # 选出得分最高的 topk_groups 个组，仅保留这些组内专家的得分
            indices = group_scores.topk(self.topk_groups, dim=-1)[1]  # 选中的组索引，形状: (bsz*seqlen, topk_groups)
            mask = torch.zeros_like(scores[..., 0]).scatter_(1, indices, True)  # 构造组掩码，被选中的组置 True
            scores = (scores * mask.unsqueeze(-1)).flatten(1)  # 非选中组的专家得分清零，展平回 (bsz*seqlen, n_routed_experts)

        # 在所有专家（或分组过滤后的专家）中选出亲和度 top-k 的专家索引
        indices = torch.topk(scores, self.topk, dim=-1)[1]  # 返回 top-k 得分对应的专家索引，形状: (bsz*seqlen, topk)
        print("Gate indices shape:", indices.shape)  # 调试输出：确认形状为 (bsz*seqlen, topk=6)
        # 根据路由索引从原始得分（未加偏置）中取出对应权重，作为各专家的加权系数
        weights = original_scores.gather(1, indices)  # gather 从 original_scores 中取每行 topk 位置的值，形状: (bsz*seqlen, topk)
        print("Gate weights :", weights)  # 调试输出：查看各 token 分配给 top-k 专家的权重值
        if self.score_func == "sigmoid":  # sigmoid 路径下各专家独立激活，topk 后权重和不为 1，需手动归一化
            weights /= weights.sum(dim=-1, keepdim=True)  # 按行归一化，使 top-k 专家的权重之和为 1，形状不变

        weights *= self.route_scale  # 乘以全局路由缩放因子，调整专家输出贡献幅度（论文中 route_scale=1.0 不改变量级）
        return weights.type_as(x), indices  # 返回 (路由权重, 专家索引)；weights 转换为与输入 x 相同的数据类型


class Expert(nn.Module):
    """
    专家（Expert）层，用于 Mixture-of-Experts（MoE）模型。

    该模块实现了一个独立的专家网络，每个专家由三层线性变换层组成。

    属性:
        w1 (nn.Module): 线性层，从输入到隐藏层的变换。
        w2 (nn.Module): 线性层，从隐藏层到输出的变换。
        w3 (nn.Module): 额外的线性层，用于特征变换。
    """
    def __init__(self, dim: int, inter_dim: int):
        """
        初始化 Expert 层。

        参数:
            dim (int): 输入和输出的维度。
            inter_dim (int): 隐藏层的维度。
        """
        super().__init__()  # 调用 nn.Module 基类构造器
        self.w1 = nn.Linear(dim, inter_dim)  # 门控分支 W₁：dim → moe_inter_dim；使用标准 nn.Linear（专家独立，不并行）
        self.w2 = nn.Linear(inter_dim, dim)  # 输出投影 W₂：moe_inter_dim → dim；将激活后特征映射回隐藏层维度
        self.w3 = nn.Linear(dim, inter_dim)  # 值分支 W₃：dim → moe_inter_dim；与 silu(w1(x)) 逐元素相乘实现 SwiGLU 门控

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        Expert 层的前向计算。

        参数:
            x (torch.Tensor): 输入张量，形状为 (count_i, dim)；count_i 为本 batch 中被路由到当前专家的 token 数量，
                由调用方 MoE.forward 通过 x[idx] 索引筛选得到，远小于 bsz*seqlen（稀疏激活）。

        返回:
            torch.Tensor: 经过 Expert 计算后的输出张量，形状为 (count_i, dim)。
        """
        return self.w2(F.silu(self.w1(x)) * self.w3(x))  # SwiGLU 激活：silu(w1(x)) 门控 × w3(x) 值，再经 w2 投影；输出形状: (count_i, dim)


## 六、完整 Transformer 架构

将前面所有基础模块（MLA、MoE、MLP、RMSNorm 等）组装为完整的 DeepSeek V3 推理模型，并通过随机输入验证前向传播的正确性。

### 6.1 MoE 混合专家模块、Transformer 块（Block）与完整模型验证

- **`MoE`**：聚合 Gate + Expert + 共享专家（Shared Expert），每次仅激活 top-k 个路由专家 + 全部共享专家，最终输出两者之和
- **`Block`**：标准 Pre-Norm Transformer 层 = RMSNorm → MLA → 残差 + RMSNorm → FFN(MLP/MoE) → 残差
- **`Transformer`**：完整模型，前 `n_dense_layers` 层使用稠密 MLP，其余层使用稀疏 MoE
- **验证**：随机生成形状 `(2, 128)` 的 token 序列，确认模型可正常前向传播，输出 logits 形状为 `(2, vocab_size)`

In [ ]:
import torch                          # PyTorch 深度学习框架主模块
import torch.nn as nn                 # 神经网络模块，提供 Module、ModuleList、Parameter 等基类
import torch.distributed as dist     # 分布式训练通信原语（all_reduce、all_gather 等）
from typing import Optional          # 类型注解：Optional[X] 表示该值可为 X 或 None

class MoE(nn.Module):
    """
    MoE（Mixture-of-Experts，专家混合）模块，用于选择性地激活多个专家网络，提高计算效率。

    主要属性：
        dim (int): 输入特征的维度。
        n_routed_experts (int): 该模型中的总专家数量。
        n_local_experts (int): 在分布式环境中，每个设备处理的专家数量。
        n_activated_experts (int): 每个输入激活的专家数量。
        gate (nn.Module): 用于计算输入到各专家的分配权重的门控机制。
        experts (nn.ModuleList): 专家网络列表，每个专家都是一个神经网络模块。
        shared_experts (nn.Module): 共享专家网络，对所有输入均生效。
    """
    def __init__(self, args: ModelArgs):
        """
        初始化 MoE 模块。

        参数：
            args (ModelArgs): 包含 MoE 参数的模型配置。
        """
        super().__init__()  # 调用 nn.Module 基类构造器
        self.dim = args.dim  # 输入/输出特征维度，类型: int；例如 2048

        # 确保专家数量可以被世界大小整除，保证每张 GPU 分配到相同数量的专家
        assert args.n_routed_experts % world_size == 0, f"专家数量必须被 world_size 整除 (world_size={world_size})"  # 否则无法均等分配，抛出 AssertionError

        self.n_routed_experts = args.n_routed_experts        # 全局路由专家总数，类型: int；例如 16（论文为 256）
        self.n_local_experts = args.n_routed_experts // world_size  # 当前 GPU 负责的专家数，类型: int；单机时等于 n_routed_experts
        self.n_activated_experts = args.n_activated_experts  # 每个 token 每次激活（路由到）的专家数，类型: int；例如 6

        # 计算当前 GPU 负责管理的专家索引区间 [experts_start_idx, experts_end_idx)
        self.experts_start_idx = rank * self.n_local_experts          # 当前 GPU 负责的专家起始索引，类型: int
        self.experts_end_idx = self.experts_start_idx + self.n_local_experts  # 当前 GPU 负责的专家结束索引（不含），类型: int

        # 初始化门控网络，负责为每个 token 计算到各专家的路由权重
        self.gate = Gate(args)  # 类型: Gate；输入 (bsz*seqlen, dim)，输出 (weights, indices) 各形状 (bsz*seqlen, topk)

        # 仅实例化当前 GPU 负责的专家，其余位置设为 None（节省显存）；ModuleList 长度仍为 n_routed_experts
        self.experts = nn.ModuleList(  # 专家列表，长度为 n_routed_experts；本 GPU 之外的槽位填 None 节省显存
            [Expert(args.dim, args.moe_inter_dim) if self.experts_start_idx <= i < self.experts_end_idx  # 属于本 GPU 范围则实例化 Expert，否则置 None
             else None for i in range(self.n_routed_experts)])  # 类型: nn.ModuleList，本 GPU 负责索引 [experts_start_idx, experts_end_idx) 的专家

        # 共享专家：对所有 token 均激活，中间维度 = n_shared_experts * moe_inter_dim（小于路由专家）
        self.shared_experts = MLP(args.dim, args.n_shared_experts * args.moe_inter_dim)  # 类型: MLP，输入/输出维度均为 dim

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        MoE 模块的前向传播。

        参数：
            x (torch.Tensor): 输入张量。

        返回：
            torch.Tensor: 经过专家计算后的输出张量。
        """
        shape = x.size()  # 保存输入原始形状 (bsz, seqlen, dim)，用于最终恢复 batch 维度
        print(f"MoE x shape: {x.shape}")  # 调试输出：确认形状为 (batch_size, seq_len, dim)
        x = x.view(-1, self.dim)  # 将 batch 和 seq_len 展平为一维：(bsz*seqlen, dim)，每行是一个 token 的特征向量

        # 通过门控机制为每个 token 选出 top-k 个专家及其权重
        weights, indices = self.gate(x)  # weights: (bsz*seqlen, topk)，indices: (bsz*seqlen, topk)，均为路由决策结果
        y = torch.zeros_like(x)  # 初始化路由专家输出累加张量，形状: (bsz*seqlen, dim)，初始全零

        # 统计每个专家被分配到的 token 总数（用于跳过未激活的专家，节省计算）
        counts = torch.bincount(indices.flatten(), minlength=self.n_routed_experts).tolist()  # 形状: (n_routed_experts,)，类型: List[int]

        for i in range(self.experts_start_idx, self.experts_end_idx):  # 仅遍历本 GPU 负责的专家索引区间，其余 GPU 的专家不参与本轮计算
            if counts[i] == 0:  # counts[i]=0 表示本 batch 没有 token 路由到第 i 号专家，无需计算
                continue  # 跳过本次循环，避免无效的空张量计算
            expert = self.experts[i]  # 获取第 i 号专家网络实例，类型: Expert
            # 找出所有路由到第 i 号专家的 token：idx 为 token 的行号，top 为该 token 的 top-k 排名中的位置
            idx, top = torch.where(indices == i)  # idx: 被选中的 token 索引，top: 对应的 topk 位置，均形状: (count_i,)
            print(f'moe-idx{idx}')  # 调试输出：打印哪些 token 被路由到第 i 号专家
            # 将第 i 号专家的计算结果按门控权重加权，累加到对应 token 的输出向量上
            y[idx] += expert(x[idx]) * weights[idx, top, None]  # expert(x[idx]) 形状: (count_i, dim)；weights[idx,top,None] 形状: (count_i, 1)；广播相乘后累加

        # 共享专家对所有 token 都计算，不受路由影响，保证模型基线能力
        z = self.shared_experts(x)  # 形状: (bsz*seqlen, dim)，所有 token 都经过共享专家

        if world_size > 1:  # 多 GPU 场景：每张 GPU 只计算了分配给自己的专家子集，需聚合得到完整路由专家输出
            dist.all_reduce(y)  # 原地加法规约：将所有 GPU 的路由专家输出 y 相加，每张 GPU 得到全局汇总的 y
        print(f"MoE y shape: {y.shape}, MoE z shape: {z.shape}")  # 调试输出：确认 y 和 z 形状均为 (bsz*seqlen, dim)
        return (y + z).view(shape)  # 路由专家输出 y 与共享专家输出 z 相加后，恢复原始形状 (bsz, seqlen, dim)




import torch                          # PyTorch 深度学习框架主模块（Block/Transformer 中复用）
import torch.nn as nn                 # 神经网络模块
import torch.distributed as dist     # 分布式训练通信原语
from typing import Optional          # 类型注解

class Block(nn.Module):
    """
    Transformer 块，结合了注意力机制和前馈网络。

    属性:
        attn (nn.Module): 多头注意力（MLA, Multi-Head Attention）。
        ffn (nn.Module): 前馈神经网络（MLP 或 MoE）。
        attn_norm (nn.Module): 注意力层的归一化层。
        ffn_norm (nn.Module): 前馈网络的归一化层。
    """
    def __init__(self, layer_id: int, args: ModelArgs):
        """
        初始化 Transformer 块。

        参数:
            layer_id (int): 该层在 Transformer 中的索引。
            args (ModelArgs): 包含 Transformer 相关参数的配置对象。
        """
        super().__init__()  # 调用 nn.Module 基类构造器
        self.attn = MLA(args)  # 多头潜在注意力机制，类型: MLA；处理自注意力计算及 KV Cache
        # layer_id < n_dense_layers（默认1）的层使用稠密 MLP，其余层使用稀疏 MoE（论文前3层为稠密）
        self.ffn = MLP(args.dim, args.inter_dim) if layer_id < args.n_dense_layers else MoE(args)  # 类型: MLP 或 MoE
        self.attn_norm = RMSNorm(args.dim)  # 注意力前的 Pre-Norm 归一化层，输入/输出形状均为 (bsz, seqlen, dim)
        self.ffn_norm = RMSNorm(args.dim)   # FFN 前的 Pre-Norm 归一化层，输入/输出形状均为 (bsz, seqlen, dim)

    def forward(self, x: torch.Tensor, start_pos: int, freqs_cis: torch.Tensor, mask: Optional[torch.Tensor]) -> torch.Tensor:
        """
        Transformer 块的前向传播。

        参数:
            x (torch.Tensor): 输入张量。
            start_pos (int): 序列的起始位置。
            freqs_cis (torch.Tensor): 预计算的旋转嵌入复指数值。
            mask (Optional[torch.Tensor]): 掩码张量，用于排除特定位置的注意力。

        返回:
            torch.Tensor: 经过 Transformer 块计算后的输出张量。
        """
        x = x + self.attn(self.attn_norm(x), start_pos, freqs_cis, mask)  # Pre-Norm → MLA → 残差连接，形状始终为 (bsz, seqlen, dim)
        x = x + self.ffn(self.ffn_norm(x))  # Pre-Norm → MLP/MoE → 残差连接，形状始终为 (bsz, seqlen, dim)
        return x  # 返回本 Block 的输出，作为下一个 Block 的输入，形状: (bsz, seqlen, dim)


class Transformer(nn.Module):
    """
    Transformer 模型，包括嵌入层、多层 Transformer 块、最终归一化层和输出层。

    属性:
        max_seq_len (int): 最大序列长度。
        embed (nn.Module): 词嵌入层。
        layers (torch.nn.ModuleList): Transformer 块的列表。
        norm (nn.Module): 所有 Transformer 层之后的归一化层。
        head (nn.Module): 输出投影层，将隐藏状态映射到词汇表大小。
        freqs_cis (torch.Tensor): 预计算的旋转嵌入复指数值。
    """
    def __init__(self, args: ModelArgs):
        """
        初始化 Transformer 模型。

        参数:
            args (ModelArgs): 包含 Transformer 相关参数的配置对象。
        """
        global world_size, rank  # 声明修改全局变量，确保后续所有模块使用最新的分布式信息
        world_size = dist.get_world_size() if dist.is_initialized() else 1  # 获取分布式训练的总进程数（GPU 数），单机为 1
        rank = dist.get_rank() if dist.is_initialized() else 0              # 获取当前进程的 rank 编号，单机为 0
        Linear.dtype = torch.float8_e4m3fn if args.dtype == "fp8" else torch.bfloat16  # 根据配置设置全局线性层权重精度：FP8 或 BF16
        super().__init__()  # 调用 nn.Module 基类构造器（在全局变量设置之后调用，确保子模块构建时已知 dtype）
        self.max_seq_len = args.max_seq_len  # 模型支持的最大序列长度，类型: int
        self.embed = ParallelEmbedding(args.vocab_size, args.dim)  # 并行词嵌入层，将 token id 转换为 dim 维向量，输出形状: (bsz, seqlen, dim)
        self.layers = torch.nn.ModuleList()  # 空的 ModuleList，用于存放所有 Transformer Block，类型: nn.ModuleList
        for layer_id in range(args.n_layers):  # 按顺序构建 n_layers 个 Transformer Block（索引从 0 到 n_layers-1）
            self.layers.append(Block(layer_id, args))  # 追加 Block；layer_id < n_dense_layers 时为 MLP，否则为 MoE
        self.norm = RMSNorm(args.dim)  # 最终归一化层，在 head 投影前对隐藏状态归一化，输出形状不变: (bsz, dim)
        self.head = ColumnParallelLinear(args.dim, args.vocab_size, dtype=torch.get_default_dtype())  # 语言模型头，将 dim 维隐藏向量映射到词表大小的 logits
        self.register_buffer("freqs_cis", precompute_freqs_cis(args), persistent=False)  # 预注册位置编码缓冲区，形状: (max_seq_len, qk_rope_head_dim//2)，不参与参数保存

    @torch.inference_mode()
    def forward(self, tokens: torch.Tensor, start_pos: int = 0):
        """
        Transformer 的前向传播。

        参数:
            tokens (torch.Tensor): 形状为 (batch_size, seq_len) 的输入 token ID。
            start_pos (int, 可选): 序列的起始位置，默认为 0。

        返回:
            torch.Tensor: 形状为 (batch_size, vocab_size) 的 logits。
        """
        seqlen = tokens.size(1)  # 获取输入序列长度，类型: int；例如 128
        h = self.embed(tokens)  # 词嵌入，将 token id 映射为向量，输出形状: (bsz, seqlen, dim)
        freqs_cis = self.freqs_cis[start_pos:start_pos+seqlen]  # 取当前序列对应位置的 RoPE 编码，形状: (seqlen, qk_rope_head_dim//2)
        mask = None  # 默认无掩码（推理时单 token 自回归续写不需要因果掩码）
        if seqlen > 1:  # prefill 阶段（处理多个 token）需要因果掩码，防止 token 看到未来位置信息
            mask = torch.full((seqlen, seqlen), float("-inf"), device=tokens.device).triu_(1)  # 上三角置 -inf 的因果掩码，形状: (seqlen, seqlen)
        for layer in self.layers:  # 依次通过所有 Transformer Block，实现逐层特征提取
            h = layer(h, start_pos, freqs_cis, mask)  # h 形状始终为 (bsz, seqlen, dim)，每层做注意力+FFN变换
        h = self.norm(h)[:, -1]  # 取最后一个时间步的隐藏状态：归一化 (bsz,seqlen,dim) → 取最后步 → (bsz, dim)
        logits = self.head(h)  # 投影到词表空间，输出 logits，形状: (bsz, vocab_size/world_size)（列并行，单机时为完整 vocab_size）
        if world_size > 1:  # 列并行 head 只计算了部分 vocab 列，需 all_gather 拼接得到完整 logits
            all_logits = [torch.empty_like(logits) for _ in range(world_size)]  # 为每个进程分配接收缓冲区，类型: List[Tensor]，长度: world_size
            dist.all_gather(all_logits, logits)  # 将各 GPU 的局部 logits 汇聚到列表（每个元素形状: (bsz, vocab_size/world_size)）
            logits = torch.cat(all_logits, dim=-1)  # 沿词表维度拼接，得到完整 logits，形状: (bsz, vocab_size)
        return logits  # 返回最终 logits，形状: (bsz, vocab_size)，用于计算 next-token 预测


if __name__ == "__main__":  # 仅在直接运行本脚本时执行（作为模块导入时跳过），用于单机功能验证
    torch.set_default_dtype(torch.bfloat16)   # 将全局默认张量类型设为 bfloat16，节省显存同时保持数值范围
    torch.set_default_device("cuda")           # 将默认计算设备设为 GPU，后续 torch.empty/zeros 等自动分配到显存
    torch.manual_seed(0)                       # 固定随机种子为 0，保证每次运行结果可复现
    args = ModelArgs()                         # 使用默认超参数实例化配置对象，类型: ModelArgs
    x = torch.randint(0, args.vocab_size, (2, 128))  # 随机生成 batch_size=2、seq_len=128 的 token 索引，形状: (2, 128)，类型: torch.int64
    model = Transformer(args)                  # 根据配置实例化完整 Transformer 模型，类型: Transformer
    print(model(x).size())                     # 前向传播并打印输出 logits 形状：(2, vocab_size=102400)；只输出最后一步的原因是 norm 后取了 h[:, -1]，便于练习分类任务



q shape: torch.Size([2, 128, 16, 192])
q_nope shape: torch.Size([2, 128, 16, 128]), q_pe shape: torch.Size([2, 128, 16, 64])
kv shape: torch.Size([2, 128, 576])
kv shape: torch.Size([2, 128, 512]), k_pe shape: torch.Size([2, 128, 64])
q shape: torch.Size([2, 128, 16, 192])
q_nope shape: torch.Size([2, 128, 16, 128]), q_pe shape: torch.Size([2, 128, 16, 64])
kv shape: torch.Size([2, 128, 576])
kv shape: torch.Size([2, 128, 512]), k_pe shape: torch.Size([2, 128, 64])
MoE x shape: torch.Size([2, 128, 2048])
Gate x shape: torch.Size([256, 2048])
Gate indices shape: torch.Size([256, 6])
Gate weights : tensor([[0.0625, 0.0625, 0.0625, 0.0625, 0.0625, 0.0625],
        [0.0625, 0.0625, 0.0625, 0.0625, 0.0625, 0.0625],
        [0.0625, 0.0625, 0.0625, 0.0625, 0.0625, 0.0625],
        ...,
        [0.0625, 0.0625, 0.0625, 0.0625, 0.0625, 0.0625],
        [0.0625, 0.0625, 0.0625, 0.0625, 0.0625, 0.0625],
        [0.0625, 0.0625, 0.0625, 0.0625, 0.0625, 0.0625]], device='cuda:0',
       dtype=torc